# 05 : Secrets & Security Lab

Do not hard-code credentials.

```text
Bad:
password = "secret"

Good:
identity / storage credential / secret reference
```


In [0]:
import sys
import os
from pathlib import Path

marker = "RETAIL_LAB.json"
cwd = Path(os.getcwd()).resolve()
PROJECT_ROOT = None

for candidate in [cwd, *cwd.parents]:
    if (candidate / marker).exists():
        PROJECT_ROOT = str(candidate)
        break

if PROJECT_ROOT is None:
    for search_root in [Path("/Workspace/Users"), Path("/Workspace/Shared"), Path("/Workspace")]:
        if not search_root.exists():
            continue
        for current_root, dirs, files in os.walk(search_root):
            dirs[:] = [d for d in dirs if not d.startswith(".") and d not in {"Trash", ".Trash"}]
            if marker in files:
                PROJECT_ROOT = current_root
                break
        if PROJECT_ROOT:
            break

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Project not found. Import the complete ZIP into Databricks Workspace."
    )

SRC_ROOT = f"{PROJECT_ROOT}/src"
if SRC_ROOT not in sys.path:
    sys.path.insert(0, SRC_ROOT)

from bootstrap import ensure_lab

ctx = ensure_lab(
    spark=spark,
    project_root=PROJECT_ROOT,
)

print("Project root :", PROJECT_ROOT)
print("Catalog      :", ctx.catalog)
print("Schema       :", ctx.schema)
print("Volume       :", ctx.volume)
print("Source root  :", ctx.source_root)


In [0]:
print('DO NOT DO: password = "MyRealPassword"')
print('REFERENCE PATTERN: dbutils.secrets.get(scope="my-app", key="password")')


In [0]:
try:
    scopes = dbutils.secrets.listScopes()
    print("Secret scopes visible:")
    for scope in scopes:
        print(" -", scope.name)
except Exception as exc:
    print("Secret-scope listing is not available/configured here.")
    print("That is acceptable for this training environment.")
    print("Reason:", str(exc)[:500])


### Preferred ADLS governance pattern

```text
Managed Identity / Service Principal
        ↓
Storage Credential
        ↓
External Location
        ↓
External Volume / Table
```
